In [3]:
import pandas as pd
from moviedata.core.database import pg_connect

In [27]:
sql_query = """
    -- Select crew by average moview rating and appearance count
    WITH trusted_crew AS (
        SELECT role, name, AVG(rating) AS val, COUNT(*) AS cnt
        FROM "1_bronze".ltboxd__ratings_and_crew
        GROUP BY role, name
        HAVING AVG(rating) >= 3.5 AND COUNT(*) > 2
    )
    SELECT rc.*
    FROM "1_bronze".ltboxd__ratings_and_crew rc
    JOIN trusted_crew tc 
        ON rc.role = tc.role 
    AND rc.name = tc.name;
"""

pg = pg_connect()
df = pg.sql_query(sql_query)

cnt_full = df.shape[0]
cnt_peop = df['name'].nunique()

print(f"{cnt_full} rows total. {cnt_peop} distinct people! Ratio: {round(cnt_full/cnt_peop)}:1")

df

7230 rows total. 1521 distinct people! Ratio: 5:1


,movie_name,movie_year,rating,role,name
0,10 Cloverfield Lane,2016,4.0,Hairstyling,Teressa Hill
1,10 Cloverfield Lane,2016,4.0,Makeup,Matthew W. Mungle
2,10 Cloverfield Lane,2016,4.0,Sound,Gadou Naudin
3,10 Cloverfield Lane,2016,4.0,Stunts,Lex D. Geddings
4,10 Cloverfield Lane,2016,4.0,Stunts,Thirl Haston
...,...,...,...,...,...
7225,Zombieland,2009,4.0,Stunts,Douglas Crosby
7226,Zombieland,2009,4.0,Stunts,Lex D. Geddings
7227,Zombieland,2009,4.0,Stunts,Drew Leary
7228,Zombieland,2009,4.0,Stunts,Steven Ritzi


In [23]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Assuming your DataFrame 'df' is already loaded with columns:
# movie_name, movie_year, rating, role, name

# --- Step 1: Reshape data from Long format to Wide format (One-Row per Movie) ---
# Since a movie has multiple crew members across multiple rows, we group by movie_name
# and collect all crew names into a list.
movie_dataset = (
    df.groupby('movie_name')
    .agg(
        rating=(
            'rating',
            'first',
        ),  # Rating is constant per movie across rows
        crew_members=(
            'name',
            lambda x: list(x.dropna()),
        ),  # Collect names into a list
    )
    .reset_index()
)

# --- Step 2: One-Hot Encode the Crew Members ---
# No matter how many people you throw into that list—whether it's just 1 person or 20—the binarizer
# will handle it seamlessly.
# If a person wasn't in the training set, it is simply ignored.
mlb = MultiLabelBinarizer()
X_encoded = mlb.fit_transform(movie_dataset['crew_members'])

# Convert the resulting numpy array into a DataFrame with person names as columns
X = pd.DataFrame(X_encoded, columns=mlb.classes_)
y = movie_dataset['rating']

# --- Step 3: Train / Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Step 4: Fit the Decision Tree Regressor ---
# Note: max_depth is set to prevent the tree from overfitting completely to the training set.
model = DecisionTreeRegressor(max_depth=6, random_state=42)
# model = RandomForestRegressor(max_depth=6, random_state=42)
model.fit(X_train, y_train)

# --- Step 5: Predictions & Evaluation ---
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("--- Model Evaluation Metrics ---")
print(f"R-squared (R2): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

--- Model Evaluation Metrics ---
R-squared (R2): -0.0101
Mean Absolute Error (MAE): 0.6439
Root Mean Squared Error (RMSE): 0.8387


In [12]:
# 1. Get the top 10 crew members by feature importance
top_10_people = (
    pd.DataFrame({'person': X.columns, 'importance': model.feature_importances_})
    .sort_values(by='importance', ascending=False)
    .head(10)
)

# 2. Find all unique roles associated with these top 10 people from your original dataframe
top_10_roles = (
    df[df['name'].isin(top_10_people['person'])]
    .groupby('name')['role']
    .agg(lambda x: ', '.join(sorted(x.unique())))  # Combines multiple roles if they have any
    .reset_index()
)

# 3. Merge the importance scores back with their roles for a decluttered view
top_10_summary = (
    pd.merge(top_10_people, top_10_roles, left_on='person', right_on='name')
    .drop(columns=['name'])
    .sort_values(by='importance', ascending=False)
)

# 4. Display the clean summary table
print('--- Top 10 Crew Members: Importance & Roles ---')
print(
    top_10_summary[['person', 'role', 'importance']].to_string(
        index=False, formatters={'importance': '{:.4f}'.format}
    )
)

--- Top 10 Crew Members: Importance & Roles ---
               person           role importance
         Phil Heywood          Sound     0.1814
     Denis Villeneuve       Director     0.1592
         David Parker          Sound     0.1154
Richard Francis-Bruce         Editor     0.0901
   Eddie J. Fernandez         Stunts     0.0861
             Jina Jay        Casting     0.0832
       Nicolas Becker          Sound     0.0494
        Richard Hymns          Sound     0.0415
            Al Nelson          Sound     0.0413
           John Knoll Visual Effects     0.0252


In [13]:
# Check the actual average rating for your top important people
top_people_names = top_10_summary['person'].tolist()

actual_performance = (
    df[df['name'].isin(top_people_names)]
    .groupby(['name', 'role'])
    .agg(
        movies_counted=('movie_name', 'count'),
        avg_personal_rating=('rating', 'mean'),
    )
    .reset_index()
    .sort_values(by='avg_personal_rating', ascending=False)
)

print(actual_performance)

                    name            role  movies_counted  avg_personal_rating
2       Denis Villeneuve        Director               7             4.428571
7           Phil Heywood           Sound               6             4.250000
8  Richard Francis-Bruce          Editor               6             4.166667
3     Eddie J. Fernandez          Stunts               6             4.000000
1           David Parker           Sound              13             3.961538
0              Al Nelson           Sound              12             3.916667
4               Jina Jay         Casting              18             3.833333
9          Richard Hymns           Sound              14             3.571429
5             John Knoll  Visual Effects              12             3.500000
6         Nicolas Becker           Sound              10             3.500000
